# Vinculación probabilística Censo - RRAA

In [ ]:
from splink.duckdb.linker import DuckDBLinker
from tqdm import tqdm

# Forzar que todas las barras de progreso se deshabiliten
tqdm.__init__ = lambda *args, **kwargs: None

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from splink.duckdb.linker import DuckDBLinker
import pandas as pd

os.chdir(r'C:\Users\Feder\Desktop\vinculacion_censo_rraa')

df_l = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraaa\data\raw\df_l_censo_sin_snsa.parquet')
df_r = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraaa\data\raw\df_r_rraa_sin_snsa.parquet')

#rraa = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraaa\data\raw\rraa.parquet')
censo = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraaa\data\raw\censo.parquet')
extranjeros_vinculados = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraaa\data\extranjeros_vinculados.parquet')

linker = DuckDBLinker([df_l, df_r])
linker.load_model(r'C:\Users\Feder\Desktop\vinculacion_censo_rraaa\output\modelos\modelo4.json')

Se parte de las siguientes dos tablas:

- Censo (3.138.813 filas)
- hub_personas_DNIC (6.128.515 filas)

y se construye un modelo para la vinculación de individuos entre las tablas. Se tienen las siguientes variables:

- documento uruguayo
- documento extranjero
- fecha de nacimiento
- primer nombre, segundo nombre, primer apellido, segundo apellido
- departamento
- sexo

En primer lugar se implementó una vinculación entre personas censadas con documento extranjero (6.895) y las personas con documento extranjero del hub_personas (11.737.275). En esa etapa se vincularon **4.589** personas de censo, las cuáles fueron excluidas del proceso utilizado para vincular al resto. El procedimiento seguido fue análogo al del proceso global, detallado a continuación

## Reglas de bloqueo

Dado que comparar todas las filas de censo con todas las de rraa sería computacionalmente inviable (se tendrían que hacer > 18.000.000.000.000 comparaciones), se seleccionan reglas de bloqueo, de forma que solo se comparan las filas que coincidan en alguna de las siguientes combinaciones de variables:

1. documento uruguayo
2. primer nombre, segundo nombre, primer apellido, segundo apellido
3. año de nacimiento, primer nombre, primer apellido
4. departamento_ajustado, año de nacimiento, primer nombre y primera letra del primer apellido
5. departamento_ajustado y fecha de nacimiento

donde en departamento_ajustado se unificaron los departamentos de Montevideo y Canelones, considerándolos como el mismo.

El siguiente gráfico presenta la cantidad de comparaciones que surgen de cada regla.

In [6]:
from splink.duckdb.linker import DuckDBLinker
from splink.duckdb.blocking_rule_library import block_on

settings = {"link_type": "link_only"}

linker_br = DuckDBLinker([df_r, df_l], settings, connection=":memory:")

br1_a = block_on(["documento"])
#br1_b = block_on(["documento_extranjero"])
#br2 = block_on(["documento", "documento_extranjero", "fecha_nacimiento"])
br3 = block_on(["fecha_nacimiento", "primer_nombre", "primer_apellido"])
#br4 = block_on(["primer_nombre", "primer_apellido"]) (433,314,151)
br5 = block_on(["primer_nombre", "segundo_nombre", "primer_apellido", "segundo_apellido"])
#br6 = block_on(["primer_nombre", "primer_apellido", "id_sexo"]) (430,033,729)
#br7 = block_on(["fecha_nacimiento", "id_sexo"]) (231,723,037)
br8 = block_on(["ano_nacimiento", "primer_nombre", "primer_apellido"])
br9 = block_on(["mes_nacimiento", "dia_nacimiento", "primer_nombre", "primer_apellido"])
#br10 = block_on(["id_departamento", "primer_nombre", "primer_apellido"]) (63,427,012)
#br11 = block_on(["id_departamento", "ano_nacimiento"]) (29,316,195,010)
br12 = block_on(["id_departamento", "fecha_nacimiento"]) #(83,471,400)
#br13 = block_on(["id_departamento", "substr(primer_nombre, 1,1)", "substr(primer_apellido, 1,1)"]) (13,581,242,609)
#br14 = block_on(["id_departamento", "primer_nombre", "ano_nacimiento"]) (292,595,639)
#br15 = block_on(["id_departamento", "substr(primer_nombre, 1,1)", "substr(primer_apellido, 1,1)", "ano_nacimiento"]) (161,287,979)
br16 = block_on(["id_departamento", "primer_nombre", "ano_nacimiento", "substr(primer_apellido, 1, 1)"])
br17 = block_on(["id_departamento_ajustado", "primer_nombre", "ano_nacimiento", "substr(primer_apellido, 1, 1)"])
br18 = block_on(["id_departamento_ajustado", "fecha_nacimiento"])

blocking_rules = [br1_a, br5, br8, br17, br18]

linker_br.cumulative_num_comparisons_from_blocking_rules_chart(blocking_rules) 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

alt.Chart(...)

## Match Weights

A continuación se presentan las comparaciones consideradas para cada variable y sus match-weights asociados.

In [7]:
linker.match_weights_chart()

alt.VConcatChart(...)

Se muestra además a modo de ejemplo los ajustes por frecuencia de palabras (cuanto menos se repita una variable en toda la base, mayor es el aporte a la vinculación cuando se encuentra un match) para la variable primer nombre. Nombres como maria o juan no aportan tanta evidencia para determinar que dos observaciones corresponden a la misma persona como otros nombres menos comunes.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

linker.tf_adjustment_chart("primer_nombre", vals_to_include="mateo")

alt.HConcatChart(...)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

linker.tf_adjustment_chart("primer_apellido")

En el caso de departamento, el valor "01" refiere a la combinación de Montevideo y Canelones. Como se puede ver, el hecho de que dos personas residan en esa región aporta poca información para determinar si son la misma, a diferencia del resto de los departamentos.

In [135]:
import warnings
warnings.filterwarnings("ignore")

linker.tf_adjustment_chart("id_departamento_ajustado")

alt.HConcatChart(...)

## Predicción

En este paso se aplica el modelo y se hacen las comparaciones respetando las reglas de bloqueo. Se genera así un puntaje (match weight) para cada vínculo y su **probabilidad de match**. 

Se asume como probabilidad mínima para considerar un match a 0.2 (aquellos vínculos que tengan una probabilidad menor se consideran no match).

Luego, para cada persona de censo se selecciona el vínculo con mayor probabilidad. La siguiente tabla muestra, para distintos umbrales o puntos de corte, cuántas personas de censo quedan vinculadas a una persona de RRAA.

In [2]:
from load_data.censo import get_personas

censo_act = get_personas(['departamento_r'])

In [ ]:
predictions_max_match_prob = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraaa\output\pred_modelo4.parquet')

censo_pred = censo_act.merge(predictions_max_match_prob, left_on = "id_censo", right_on= "unique_id_l", how = "inner")

corte2 = censo_pred[(censo_pred["match_probability"] >= 0.999)]
corte3 = censo_pred[(censo_pred["match_probability"] >= 0.9) & (censo_pred["match_probability"] < 0.999)]
corte4 = censo_pred[(censo_pred["match_probability"] < 0.9)]

def separador_miles(numbers):
    
    def separar(number):
        return "{:,}".format(number).replace(",", ".")
    
    try:
        return [separar(number) for number in numbers]
    except:
        return separar(numbers)
        
        
pd.DataFrame({
    "Probabilidad de match": ["> 0.999", "0.9 - 0.999", "0.2 - 0.9"],
    "Descripción del match": ["match confiable", "match poco confiable", "match no confiable"],
    "Cantidad de personas": separador_miles([len(corte2), len(corte3), len(corte4)]),
    "Proporción de Censo (%)": separador_miles([round(len(corte2)/len(censo)*100, 1), round(len(corte3)/len(censo)*100, 1), round(len(corte4)/len(censo)*100, 1)])
})

,Probabilidad de match,Descripción del match,Cantidad de personas,Proporción de Censo (%)
0,> 0.999,match confiable,2.971.244,94.7
1,0.9 - 0.999,match poco confiable,60.264,1.9
2,0.2 - 0.9,match no confiable,25.688,0.8


## Análisis por umbral

A continuación se presenta información para cada umbral y ejemplos de observaciones vinculadas.

Los ejemplos se presentan en el formato de Waterfall chart. En cada gráfico se tienen dos personas que fueron vinculadas, y cada "barra" muestra la comparación para cada variable. Los valores que aparecen sobre la barra son el de censo y rraa respectivamente, y la altura de la barra representa el puntaje que aporta cada comparación al match-weigth. En la barra "Final Score" se puede ver el match-weight final asociado a la comparación (es la suma de los match-weight de cada variable) en el eje de la izquierda, y la probabilidad de match asociada a ese match-weight en el eje de la derecha. El deslizador que figura abajo del gráfico permite ver distintas comparaciones seleccionadas al azar.

### Umbral >0.999

In [16]:
pd.DataFrame({
    "Documento": ["Match exacto", "Valor faltante", "Similar", "Diferente"],
    "Cantidad de personas": separador_miles(corte2["gamma_documento"].replace({2:1}).value_counts()),
    "Proporción (%)": round(corte2["gamma_documento"].replace({2:1}).value_counts(normalize=True)*100, 1)
})

,Documento,Cantidad de personas,Proporción (%)
gamma_documento,,,
3,Match exacto,2.833.492,95.4
-1,Valor faltante,126.801,4.3
1,Similar,10.354,0.3
0,Diferente,597,0.0


In [15]:
pd.DataFrame({
    "Fecha de nacimiento": ["Match exacto", "Similar",  "Diferente", "Valor faltante"],
    "Cantidad de personas": separador_miles(corte2["gamma_fecha_nacimiento"].replace({2:1}).value_counts()),
    "Proporción (%)": round(corte2["gamma_fecha_nacimiento"].replace({2:1}).value_counts(normalize=True)*100, 1)
})

,Fecha de nacimiento,Cantidad de personas,Proporción (%)
gamma_fecha_nacimiento,,,
3,Match exacto,2.848.201,95.9
1,Similar,104.591,3.5
0,Diferente,17.367,0.6
-1,Valor faltante,1.085,0.0


In [17]:
records_to_view  = corte2.sample(n=5, random_state = 13).to_dict(orient="records")
records_to_view.extend(
    corte2[corte2["gamma_documento"] == 0].sample(n=3, random_state=13).to_dict(orient="records")
    )
linker.waterfall_chart(records_to_view, filter_nulls=False)

alt.LayerChart(...)

### Umbral 0.9 - 0.999

In [18]:
pd.DataFrame({
    "Documento": ["Valor faltante", "Match exacto", "Diferente", "Similar"],
    "Cantidad de personas": separador_miles(corte3["gamma_documento"].replace({2:1}).value_counts()),
    "Proporción (%)": round(corte3["gamma_documento"].replace({2:1}).value_counts(normalize=True)*100, 1)
})

,Documento,Cantidad de personas,Proporción (%)
gamma_documento,,,
-1,Valor faltante,46.411,77.0
3,Match exacto,11.695,19.4
0,Diferente,1.126,1.9
1,Similar,1.032,1.7


In [19]:
pd.DataFrame({
    "Fecha de nacimiento": ["Match exacto", "Similar", "Diferente", "Valor faltante"],
    "Cantidad de personas": separador_miles(corte3["gamma_fecha_nacimiento"].replace({2:1}).value_counts()),
    "Proporción (%)": round(corte3["gamma_fecha_nacimiento"].replace({2:1}).value_counts(normalize=True)*100, 1)
})

,Fecha de nacimiento,Cantidad de personas,Proporción (%)
gamma_fecha_nacimiento,,,
3,Match exacto,46.948,77.9
1,Similar,9.104,15.1
0,Diferente,2.802,4.6
-1,Valor faltante,1.410,2.3


In [20]:
records_to_view  = corte3.sample(n=5, random_state = 13).to_dict(orient="records")
records_to_view.extend(
    corte3[corte3["gamma_documento"] == 0].sample(n=3, random_state=13).to_dict(orient="records")
    )
linker.waterfall_chart(records_to_view, filter_nulls=False)

alt.LayerChart(...)

### Umbral 0.2 - 0.9

In [21]:
pd.DataFrame({
    "Documento": ["Valor faltante", "Match exacto", "Diferente", "Similar"],
    "Cantidad de personas": separador_miles(corte3["gamma_documento"].replace({2:1}).value_counts()),
    "Proporción (%)": round(corte3["gamma_documento"].replace({2:1}).value_counts(normalize=True)*100, 1)
})

,Documento,Cantidad de personas,Proporción (%)
gamma_documento,,,
-1,Valor faltante,46.411,77.0
3,Match exacto,11.695,19.4
0,Diferente,1.126,1.9
1,Similar,1.032,1.7


In [22]:
pd.DataFrame({
    "Fecha de nacimiento": ["Match exacto", "Similar", "Valor faltante", "Diferente"],
    "Cantidad de personas": separador_miles(corte4["gamma_fecha_nacimiento"].replace({2:1}).value_counts()),
    "Proporción (%)": round(corte4["gamma_fecha_nacimiento"].replace({2:1}).value_counts(normalize=True)*100, 1)
})

,Fecha de nacimiento,Cantidad de personas,Proporción (%)
gamma_fecha_nacimiento,,,
3,Match exacto,16.400,63.8
1,Similar,5.556,21.6
-1,Valor faltante,1.928,7.5
0,Diferente,1.804,7.0


In [23]:
records_to_view  = corte4.sample(n=5, random_state = 13).to_dict(orient="records")
records_to_view.extend(
    corte4[corte4["gamma_documento"] == 2].sample(n=3, random_state=13).to_dict(orient="records")
    )
linker.waterfall_chart(records_to_view, filter_nulls=False)

alt.LayerChart(...)

## Reglas para considerar un match

Luego de analizar los umbrales de probabilidad se propone el siguiente criterio para considerar un match:

1. probabilidad_match > 0.999
2. probabilidad_match $\in$ (0.9, 0.999); (coincidencia_documentos = 1 y doc_unico = 1) o (coincidencia_fecha_nacimiento = 1 y primer_nombre y primer_apellido similares)

El criterio 2 indica que la probabilidad de match debe estar entre 0.9 y 0.999, y además debe cumplirse una de las siguientes condiciones:  
    a. deben coincidir los documentos y además el documento debe aparecer solo una vez en censo.  
    b. deben coincidir las fechas de nacimiento y tanto primer nombre como primer apellido deben ser similares (considerando una distancia de jaro_winkler > 0.88).

De los vínculos del umbral (0.9, 0.999), 46.781 cumplen con el criterio, por lo que si se suman a los vínculos del umbral >0.999 y a los vinculados por documento extranjero, se tiene un total de **3.022.611** match (96.3% de las personas censadas).

## No vinculados

Se considera como personas no vinculadas a aquellas que no tienen ningún vínculo cuya probabilidad sea mayor a 0.2

In [13]:
no_vinculados = pd.merge(df_l.rename(columns = {"unique_id":"unique_id_l"}), censo_pred[["unique_id_l"]], how = "left", indicator = True).merge(censo_act[["id_censo"]], how = "inner", left_on = "unique_id_l", right_on = "id_censo")

no_vinculados = no_vinculados[no_vinculados["_merge"] == "left_only"].merge(censo[["id_censo", "tipo_caso", "docextnro"]].rename(columns = {"id_censo":"unique_id_l", "docextnro":"documento_extranjero"}), how = "left").drop(columns="_merge")

In [14]:
from IPython.display import Markdown, display

display(Markdown(f"Se tienen {separador_miles(len(no_vinculados))} personas de censo que no han sido vinculadas a una persona de rraa. A continuación se presenta su distribución según tipo de caso y datos faltantes."))

Se tienen 77.031 personas de censo que no han sido vinculadas a una persona de rraa. A continuación se presenta su distribución según tipo de caso y datos faltantes.

In [15]:
tipo_caso = no_vinculados.value_counts("tipo_caso")

pd.DataFrame({
    "Tipo de caso": tipo_caso.index,
    "No vinculados": separador_miles(tipo_caso),
    "Proporción (%)": separador_miles(round(tipo_caso/len(no_vinculados)*100, 2))
})

,Tipo de caso,No vinculados,Proporción (%)
0,capi,53.449,69.39
1,cawiincompletonovarbasicas,13.183,17.11
2,cawiasociado,4.203,5.46
3,refugiosmides,1.516,1.97
4,cawisinasociar,1.453,1.89
5,registros,1.274,1.65
6,intemperiemides,740,0.96
7,capiincompletonovarbasicas,481,0.62
8,recuperacion,249,0.32
9,refugiossen,232,0.3


In [16]:
missing = round(no_vinculados.merge(df_l[["unique_id", "id_departamento"]].rename(columns = {"unique_id":"id_censo"})).isnull().sum()[["primer_nombre", "segundo_nombre", "primer_apellido", "segundo_apellido", "documento", "fecha_nacimiento", "id_departamento"]]/len(no_vinculados)*100, 2)

pd.DataFrame({
    "Proporción de valores faltantes (%)": missing
})

,Proporción de valores faltantes (%)
primer_nombre,7.48
segundo_nombre,97.63
primer_apellido,13.36
segundo_apellido,97.35
documento,91.61
fecha_nacimiento,20.42
id_departamento,0.00


Como se puede ver, gran parte de las personas que no fueron vinculadas solo cuentan con primer nombre, primer apellido, fecha de nacimiento y departamento (varias tampoco tienen fecha de nacimiento). A raíz de esto se descarta de la vinculación un grupo de personas que debido a sus datos faltantes, se consideran no vinculables. 

En este grupo están las personas que no tienen documento o el mismo no es único en la tabla de censo, y tampoco tienen dato en primer nombre o primer apellido (es decir, solo tienen uno de los dos o ninguno). Este grupo queda formado por **10.981** personas.

In [17]:
# no vinculables

documentos = df_l.rename(columns={"documento":"documento_l"})["documento_l"].value_counts()

no_vinculados["doc_unico"] = no_vinculados["documento"].apply(lambda x: 1 if pd.notna(x) and documentos.get(x, 0) == 1 else 0)

no1 = pd.isnull(no_vinculados["documento"]) & pd.isnull(no_vinculados["fecha_nacimiento"]) & pd.isnull(no_vinculados["primer_nombre"]) & pd.isnull(no_vinculados["primer_apellido"])
#no2 = (no_vinculados["doc_unico"] == 0) & ((pd.isnull(no_vinculados["fecha_nacimiento"]) & (pd.isnull(no_vinculados["primer_nombre"]) | pd.isnull(no_vinculados["primer_apellido"]))) | (pd.isnull(no_vinculados["primer_nombre"]) & pd.isnull(no_vinculados["primer_apellido"])))
no3 = (no_vinculados["doc_unico"] == 0) & ((pd.isnull(no_vinculados["primer_nombre"]) | pd.isnull(no_vinculados["primer_apellido"])))
#no3 = (no_vinculados["doc_unico"] == 0) & pd.isnull(no_vinculados["fecha_nacimiento"]) & pd.isnull(no_vinculados["primer_nombre"]) & pd.isnull(no_vinculados["primer_apellido"])

#& pd.isnull(no_vinculados["segundo_nombre"]) & pd.isnull(no_vinculados["segundo_apellido"])

no_vinculables = no_vinculados[no1 | no3]

In [116]:
vinculables = no_vinculados[~(no1 | no3)]

## arrancar con modelo solo usando nombres y fecha de nacimiento ://

In [ ]:
no_vinculados["estado_documento"] = np.where(
    pd.isnull(no_vinculados["documento"]) & pd.isnull(no_vinculados["documento_extranjero"]),
    "sin_documento",
    np.where(
        pd.isnull(no_vinculados["documento"]),
        "extranjero",
        "uruguayo"
    )
)

documentos = no_vinculados.value_counts("estado_documento").sort_index(ascending=False)

df_l = df_l.merge(censo[["id_censo", "docextnro"]].rename(columns = {"id_censo":"unique_id", "docextnro":"documento_extranjero"}), how = "left")

df_l["estado_documento"] = np.where(
    pd.isnull(df_l["documento"]) & pd.isnull(df_l["documento_extranjero"]),
    "sin_documento",
    np.where(
        pd.isnull(df_l["documento"]),
        "extranjero",
        "uruguayo"
    )
)

documentos_total = df_l.value_counts("estado_documento").sort_index(ascending=False)

pd.DataFrame({
    "Documento": documentos.index,
    "No vinculados": separador_miles(documentos),
    "Proporción sobre el total de cada grupo (%)": separador_miles(round(documentos/documentos_total*100, 2))
})

El siguiente cuadro resume el resultado del proceso de vinculación:

In [18]:
pd.DataFrame({
    "Estado": ["match", "match no considerado", "no match", "no vinculable", "TOTAL"],
    "Personas de censo": separador_miles([3022611, 39171, 66050, 10981, 3138813]),
    "Proporción (%)": separador_miles(np.round(np.array([3022841, 39174, 66051, 10982, 3139048])/3139048*100, 2))
})

,Estado,Personas de censo,Proporción (%)
0,match,3.022.611,96.3
1,match no considerado,39.171,1.25
2,no match,66.050,2.1
3,no vinculable,10.981,0.35
4,TOTAL,3.138.813,100.0


## Duplicados Censo

El proceso de vinculación también permite analizar candidatos a duplicados para censo, en la medida que dos personas se vinculan con alta probabilidad a la misma de rraa.

In [19]:
dups = predictions_max_match_prob[predictions_max_match_prob.duplicated(subset="unique_id_r", keep = False)]

dups_corte2 = corte2[corte2.duplicated(subset="unique_id_r", keep = False)]

A continuación se muestra una tabla con la cantidad de personas duplicadas según el punto de corte de la probabilidad de match (todas las personas de censo vinculadas a la persona de rraa deben superar ese umbral). El total de duplicados se va acumulando al flexibilizar la probabilidad de match, alcanzando 27.966 personas de censo duplicadas si se toman todos los vínculos con probabilidad > 0.2.

In [20]:
dups = list((len(corte[corte.duplicated(subset="unique_id_r", keep = False)].drop_duplicates(subset="unique_id_r")) for corte in [corte2, pd.concat([corte2, corte3]), pd.concat([corte2, corte3, corte4])]))

pd.DataFrame({
    "Probabilidad de match": ["> 0.999", "0.9 - 0.999", "0.2 - 0.9"],
    "Descripción del match": ["match confiable", "+ match poco confiable", "+ match no confiable"],
    "Cantidad de personas duplicadas": separador_miles(dups)
})

,Probabilidad de match,Descripción del match,Cantidad de personas duplicadas
0,> 0.999,match confiable,16.297
1,0.9 - 0.999,+ match poco confiable,22.133
2,0.2 - 0.9,+ match no confiable,27.966
